# TC_06: FCNN with noise injection perturbation
Inject noise at 10%, 20%, 50% levels. Run full diagnostic pipeline at each level and compare against TC_03 (Golden Baseline)

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_fcnn, evaluate_model_fcnn
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_tab, mc_dropout_prediction_tab
from src.inference_diagnostics.explainability import shap_tab, lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.data_diagnostics.perturbations import inject_noise_tab
from src.utils.config_loader import load_config, get_tabular_config, load_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id


config = load_config()
tracker = PerformanceTracker()

### Load clean data


In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
quality_report = check_tabular_quality(X_train, y_train, config)

# Identity for this notebook
dataset_short = get_tabular_config(config)['short_name']
model_name = 'fcnn'
test_case = 'TC06'


# Load seed
seed = config['random_seeds']['primary_seed']
noise_levels =  config['stage1_data_quality']['perturbations']['noise_injection']['noise_levels']

# Golden baseline thresholds written by TC_03
mc_threshold = load_threshold(config, dataset_short, model_name, 'mc')
de_threshold = load_threshold(config, dataset_short, model_name, 'de')
print(f"Loaded thresholds - MC: {mc_threshold}, DE: {de_threshold}")

Loading dataset.
Dataset: Diabetes Health Indicators
Duplicate rows kept (24206 found).
No missing value found.
Data loaded for 202944 train, 50736 test
 Features: 21
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 18580
Total outlier count: 89295
EDA completed for Diabetes Health Indicators
Loaded thresholds - MC: 0.0616, DE: 0.0302


### Run full pipeline for each perturbation levels

In [3]:
all_results = {}

for noise_level in noise_levels:
    print(f"Inject noise for fraction: {noise_level * 100:.0f}%")

    tracker = PerformanceTracker()

    # Inject noise
    X_train_corrupted = inject_noise_tab(X_train, noise_level, seed)
    corrupted_report = check_tabular_quality(X_train_corrupted, y_train, config)
    plot_class_distribution(corrupted_report, f"FCNN Diabetes inject noise level: {noise_level * 100:.0f}% ", config)
    plot_correlation(corrupted_report, f"FCNN Diabetes inject noise level: {noise_level * 100:.0f}%", config)
    plot_feature_boxplots(X_train_corrupted, f"FCNN Diabetes inject noise level: {noise_level * 100:.0f}%", config)

    # Train FCNN with fault values
    tracker.start_performance_track()
    fcnn_model = train_fcnn(X_train_corrupted, y_train, config)
    tracker.stop_performance_track(f"FCNN training inject noise level: {noise_level * 100:.0f}%")

    tracker.start_performance_track()
    fcnn_accuracy, fcnn_prediction, fcnn_report = evaluate_model_fcnn(fcnn_model, X_test, y_test, config)
    tracker.stop_performance_track(f"FCNN evaluation inject noise level: {noise_level * 100:.0f}%")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_tab(fcnn_model, X_test, config)
    tracker.stop_performance_track(f"FCNN MC Dropout inject noise level: {noise_level * 100:.0f}%")

    print(f"MC Dropout uncertainty status:")
    print(f"Mean: {mc_uncertainty.mean()}")
    print(f"Max: {mc_uncertainty.max()}")
    print(f" 90th percentile (threshold): {mc_threshold}")

    plot_uncertainty_distribution(mc_uncertainty, f"FCNN MC Dropout inject noise level: {noise_level * 100:.0f}% ", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_tab(train_fcnn, X_train_corrupted, y_train, X_test, config)
    tracker.stop_performance_track(f"FCNN Deep Ensemble inject noise level: {noise_level * 100:.0f}%")

    print(f"Deep Ensembl uncertainty status:")
    print(f"Mean: {de_uncertainty.mean()}")
    print(f"Max: {de_uncertainty.max()}")
    print(f" 90th percentile (threshold): {de_threshold}")

    plot_uncertainty_distribution(de_uncertainty, f"FCNN Deep Ensemble inject noise level: {noise_level * 100:.0f}%", de_threshold, config)

    # SHAP
    tracker.start_performance_track()
    shap_values, shap_samples = shap_tab(fcnn_model, X_train_corrupted, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN SHAP inject noise level: {noise_level * 100:.0f}%")
    print(f"SHAP values shape: {shap_values.shape}")

    # LIME
    tracker.start_performance_track()
    lime_explanations, lime_samples = lime_tab(fcnn_model, X_train_corrupted, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN LIME inject noise level: {noise_level * 100:.0f}%")
    print(f"LIME explanations: {len(lime_explanations)}")

    # Calculate consistency
    feature_names = list(X_train.columns)
    consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)

    # MC Dropout with XAI flags
    mc_y_predictions = mc_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    mc_flags = assign_flags(mc_uncertainty[:len(consistency_scores)], consistency_scores, mc_threshold, 0.5)
    mc_flag_results = evaluate_flags(mc_flags, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags, f"FCNN MC Dropout + XAI Flagging inject noise level: {noise_level * 100:.0f}%", config)

    # Deep Ensemble flags
    de_y_predictions = de_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    de_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, de_threshold, 0.5)
    de_flag_results = evaluate_flags(de_flags, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags, f"FCNN Deep Ensembles + XAI Flagging inject noise level: {noise_level * 100:.0f}%", config)

    # MC Dropout without XAI (constant consistency)
    con_consistency = np.ones(len(consistency_scores))
    mc_flags_uq_only = assign_flags(mc_uncertainty[:len(consistency_scores)], con_consistency, mc_threshold, 0.5)

    print(f"MC Dropout without XAI Flagging:")
    mc_only_flag_results = evaluate_flags(mc_flags_uq_only, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags_uq_only, f"FCNN MC Dropout only inject noise level: {noise_level * 100:.0f}%", config)

    print(f"MC Dropout with XAI green accuracy: {mc_flag_results['GREEN']['accuracy']}")
    print(f"MC Dropout without XAI green accuracy: {mc_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy']}")

    # Deep Ensemble without XAI (constant consistency)
    de_flags_uq_only = assign_flags(de_uncertainty[:len(consistency_scores)], con_consistency, de_threshold, 0.5)

    print(f"Deep Ensemble without XAI Flagging:")
    de_only_flag_results = evaluate_flags(de_flags_uq_only, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags_uq_only, f"FCNN Deep Ensembles only inject noise level: {noise_level * 100:.0f}%", config)

    print(f"Deep Ensembles with XAI green accuracy: {de_flag_results['GREEN']['accuracy']}")
    print(f"Deep Ensembles without XAI green accuracy: {de_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy']}")

    level_result = {
    'perturbation': 'noise injection',
    'noise_level': noise_level,
    'model': 'FCNN',
    'accuracy': round(fcnn_accuracy, 4),
    'classification_report': fcnn_report,
    'quality_report': quality_report['feature_stats'],
    'corrupted_quality_report': corrupted_report['feature_stats'],
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging_mc': mc_flag_results,
    'flagging_de': de_flag_results,
    'flagging_mc_only': mc_only_flag_results,
    'mc_vs_mc_plus_xai': round(mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy'], 4),
    'flagging_de_only': de_only_flag_results,
    'de_vs_de_plus_xai': round(de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
    }

    all_results[f"{noise_level * 100:.0f}"] = level_result

        # Per sample results for this noise levels
    experiment_id = build_experiment_id(dataset_short, model_name, test_case, f"ni{int(noise_level*100)}")
    save_per_sample(
        config,
        experiment_id,
        y_true=y_test,
        y_pred=mc_y_predictions,
        mc_uncertainty=mc_uncertainty,
        de_uncertainty=de_uncertainty,
        consistency=consistency_scores,
    )

    # Save individual report
    report_output = generate_report(config, f"{get_tabular_config(config)['name']} - FCNN noise injection {noise_level*100:.0f}%", stage1_result=level_result)
    save_report(report_output, f'{dataset_short.upper()}_TC_06_FCNN_Noise_Injection_{int(noise_level*100)}pct.json', config)


Inject noise for fraction: 10%
Injecting noise at level 0.1
Noise added to 21 
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 0
Total outlier count: 70379
EDA completed for Diabetes Health Indicators
Saved: results/figures\class_distribution_fcnn_diabetes_inject_noise_level:_10%_.png
Saved: results/figures\correlation_fcnn_diabetes_inject_noise_level:_10%.png
Saved: results/figures\boxplot_fcnn_diabetes_inject_noise_level:_10%.png
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
 Early stopping at epoch 44
FCNN training completed.
FCNN training inject noise level: 10%: Time:104.15s, Memory:306.04MB
Model evaluation started for FCNN.
{'0': {'precision': 0.868762925928157, 'recall': 0.9908397645819498, 'f1-score': 0.9257943725259441, 'support': 43667.0}, '1': {'precision': 0.5712754555198285, 'recall': 0.07539963219691612, 'f1-score': 0.13321669582604348, 'support': 7069.0

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP inject noise level: 10%: Time:7.55s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME inject noise level: 10%: Time:3.14s, Memory:13.07MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 1 Accuracy:0.0%
YELLOW: Count: 16 Accuracy:81.2%
GREEN: Count: 183 Accuracy:87.4%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_inject_noise_level:_10%.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 25 Accuracy:76.0%
GREEN: Count: 175 Accuracy:88.0%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_inject_noise_level:_10%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 5 Accuracy:60.0%
GREEN: Count: 195 Accuracy:87.2%
Flag

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP inject noise level: 20%: Time:7.66s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME inject noise level: 20%: Time:3.36s, Memory:12.21MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 23 Accuracy:69.6%
GREEN: Count: 177 Accuracy:87.6%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_inject_noise_level:_20%.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 20 Accuracy:75.0%
GREEN: Count: 180 Accuracy:87.8%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_inject_noise_level:_20%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 10 Accuracy:30.0%
GREEN: Count: 190 Accuracy:88.4%
Fla

C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` par

Epoch 10/50
Epoch 20/50
 Early stopping at epoch 29
FCNN training completed.
Training model with seed 1
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Training model with seed 2
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Training model with seed 3
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Training model with seed 4
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Deep Ensemble finished for tabular.
FCNN Deep Ensemble inject noise level: 50%: Time:484.00s, Memory:0.20MB
Deep Ensembl uncertainty status:
Mean: 0.010062793269753456
Max: 0.05269875377416611
 90th percentile (threshold): 0.0302
Saved: results/figures\uncertainty_distribution_fcnn_deep_ensemble_inject_noise_level:_50%.png
SHAP started.


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP inject noise level: 50%: Time:7.43s, Memory:0.26MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME inject noise level: 50%: Time:3.71s, Memory:0.00MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 20 Accuracy:85.0%
GREEN: Count: 180 Accuracy:86.1%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_inject_noise_level:_50%.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 20 Accuracy:95.0%
GREEN: Count: 180 Accuracy:85.0%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_inject_noise_level:_50%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 4 Accuracy:25.0%
GREEN: Count: 196 Accuracy:87.2%
Flagg